# Latent Space Alignment via Relative Representations

This notebook implements the core concept from the paper **"Relative Representations: Sharing Latent Spaces Across Lives"** (Moschella et al., 2022). 

### The Core Concept:
Different models (like **Qwen-0.5B** and **TinyLlama-1.1B**) represent the world in completely different absolute coordinates due to variations in size, architecture, and training data. 

However, the *relative relationships* between concepts (e.g., how close "market" is to "money", "company", or "business") remain invariant across these spaces. By selecting a shared set of **parallel anchor words**, we can project absolute embeddings into a **relative coordinate system** where they can be compared directly across models, languages, and dimensions.

In [1]:
!pip install transformers torch accelerate


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Set device to GPU if available (CUDA for Nvidia, MPS for Apple Silicon, or CPU)
device = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print(f"Using device: {device}\n")

# ==========================================
# 1. Load Qwen 2.5 (0.5B Instruct)
# ==========================================
print("Loading Qwen 2.5 (0.5B)...")
qwen_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    dtype=torch.float16 if device != "cpu" else torch.float32,
    device_map=device
)


# ==========================================
# Test Generation
# ==========================================
prompt = "Explain quantum computing in one sentence."

print("\n--- Testing Qwen 2.5 (0.5B) ---")
# Format with chat template for Qwen
qwen_messages = [{"role": "user", "content": prompt}]
qwen_text = qwen_tokenizer.apply_chat_template(
    qwen_messages, 
    tokenize=False, 
    add_generation_prompt=True
)
qwen_inputs = qwen_tokenizer([qwen_text], return_tensors="pt").to(device)

with torch.no_grad():
    qwen_generated_ids = qwen_model.generate(
        **qwen_inputs, 
        max_new_tokens=50,
        temperature=0.7,
        do_sample=True
    )
# Extract only the newly generated tokens
qwen_generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(qwen_inputs.input_ids, qwen_generated_ids)
]
print(qwen_tokenizer.batch_decode(qwen_generated_ids, skip_special_tokens=True)[0])



/Users/abdullah/Pytorch/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/abdullah/Pytorch/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps

Loading Qwen 2.5 (0.5B)...


`torch_dtype` is deprecated! Use `dtype` instead!



--- Testing Qwen 2.5 (0.5B) ---
Quantum computers utilize the principles of quantum mechanics to perform calculations on a vast number of possibilities simultaneously, potentially offering exponential speedups over classical computers for certain tasks.


In [3]:
# ==========================================
# 2. Load TinyLlama (1.1B Chat)
# ==========================================
print("Loading TinyLlama (1.1B)...")
tinyllama_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tiny_tokenizer = AutoTokenizer.from_pretrained(tinyllama_model_name)
tiny_model = AutoModelForCausalLM.from_pretrained(
    tinyllama_model_name,
    dtype=torch.float16 if device != "cpu" else torch.float32,
    device_map=device
)

print("\n--- Testing TinyLlama (1.1B) ---")
# Format with chat template for TinyLlama
tiny_messages = [{"role": "user", "content": prompt}]
tiny_text = tiny_tokenizer.apply_chat_template(
    tiny_messages, 
    tokenize=False, 
    add_generation_prompt=True
)
tiny_inputs = tiny_tokenizer([tiny_text], return_tensors="pt").to(device)

with torch.no_grad():
    tiny_generated_ids = tiny_model.generate(
        **tiny_inputs, 
        max_new_tokens=50,
        temperature=0.7,
        do_sample=True
    )
# Extract only the newly generated tokens
tiny_generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(tiny_inputs.input_ids, tiny_generated_ids)
]
print(tiny_tokenizer.batch_decode(tiny_generated_ids, skip_special_tokens=True)[0])


Loading TinyLlama (1.1B)...

--- Testing TinyLlama (1.1B) ---
Quantum computing is a type of computing that uses quantum mechanics to process data. It involves the use of quantum bits (qubits), which are capable of simultaneously holding both 0 and 1 states, to perform complex calculations faster than classical computers


In [4]:
english_anchors = [
    "time", "year", "people", "way", "day", "man", "thing", "woman", "life", "child",
    "world", "school", "state", "family", "student", "group", "country", "problem", "hand", "part",
    "place", "case", "week", "company", "system", "program", "question", "work", "government", "number",
    "night", "point", "home", "water", "room", "mother", "area", "money", "story", "fact",
    "month", "lot", "right", "study", "book", "eye", "job", "word", "business", "issue",
    "side", "kind", "head", "house", "service", "friend", "father", "power", "hour", "game",
    "line", "end", "member", "law", "car", "city", "community", "name", "idea", "body",
    "information", "back", "parent", "face", "others", "level", "office", "door", "health", "person",
    "art", "war", "history", "party", "result", "change", "morning", "reason", "research", "girl",
    "guy", "moment", "air", "teacher", "force", "education", "foot", "boy", "age", "policy"
]

In [5]:
chinese_anchors = [
    "时间", "年", "人", "方式", "天", "男人", "事情", "女人", "生命", "孩子",
    "世界", "学校", "国家", "家庭", "学生", "团队", "国家", "问题", "手", "部分",
    "地方", "案件", "星期", "公司", "系统", "项目", "问题", "工作", "政府", "数字",
    "夜晚", "观点", "家", "水", "房间", "母亲", "区域", "金钱", "故事", "事实",
    "月份", "许多", "权利", "研究", "书", "眼睛", "工作", "词语", "商业", "议题",
    "面", "种类", "头", "房子", "服务", "朋友", "父亲", "力量", "小时", "游戏",
    "线", "结束", "成员", "法律", "汽车", "城市", "社区", "名字", "想法", "身体",
    "信息", "背部", "父母", "脸", "其他人", "水平", "办公室", "门", "健康", "人",
    "艺术", "战争", "历史", "党派", "结果", "改变", "早晨", "原因", "研究", "女孩",
    "家伙", "时刻", "空气", "老师", "力量", "教育", "脚", "男孩", "年龄", "政策"
]

In [6]:
english_test = [
    "market", "guide", "source", "future", "nature", 
    "action", "growth", "growth", "signal", "theory"
]

In [7]:
chinese_test= [
    "市场", "指南", "来源", "未来", "自然", 
    "行动", "增长", "选择", "信号", "理论"
]

In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Fix: Correctly check for Apple Silicon GPU (MPS)
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct").to(device)
model.eval()


chinese_embeddings_anchors = []
chinese_embeddings_test = []

with torch.no_grad():
    # Process Anchor Words
    for word in chinese_anchors:
        inputs = tokenizer(word, return_tensors="pt").to(device)
        
        # Optimization: use model.model to get only the last hidden states directly 
        outputs = model.model(**inputs)
        last_hidden = outputs.last_hidden_state  # (batch=1, seq_len, hidden_dim)
        
        # Mean pool across the sequence length dimension
        embedding = last_hidden.mean(dim=1).squeeze(0)  # (hidden_dim,)
        chinese_embeddings_anchors.append(embedding.cpu())
        
    # Process Test Words
    for word in chinese_test:
        inputs = tokenizer(word, return_tensors="pt").to(device)
        
        # Optimization: same base-model forward pass
        outputs = model.model(**inputs)
        last_hidden = outputs.last_hidden_state  # (batch=1, seq_len, hidden_dim)
        
        embedding = last_hidden.mean(dim=1).squeeze(0)  # (hidden_dim,)
        chinese_embeddings_test.append(embedding.cpu())

# Now you have lists of PyTorch tensors on CPU, ready for alignment!
print(f"Loaded {len(chinese_embeddings_anchors)} anchor embeddings of shape {chinese_embeddings_anchors[0].shape}")


Using device: mps
Loaded 100 anchor embeddings of shape torch.Size([896])


In [13]:
# Stack the list of 1D tensors [hidden_dim] into a 2D tensor [num_words, hidden_dim]
anchor_tensor = torch.stack(chinese_embeddings_anchors)
test_tensor = torch.stack(chinese_embeddings_test)

# Organize them into a dictionary
embeddings_dict = {
    "anchors": anchor_tensor,
    "anchor_words": chinese_anchors,
    "test": test_tensor,
    "test_words": chinese_test
}

# Save to disk
torch.save(embeddings_dict, "chinese_embeddings.pt")
print("Saved embeddings successfully to 'chinese_embeddings.pt'!")


Saved embeddings successfully to 'chinese_embeddings.pt'!


In [14]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Fix: Correctly check for Apple Silicon GPU (MPS)
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0").to(device)
model.eval()


english_embeddings_anchors = []
english_embeddings_test = []

with torch.no_grad():
    # Process Anchor Words
    for word in english_anchors:
        inputs = tokenizer(word, return_tensors="pt").to(device)
        
        # Optimization: use model.model to get only the last hidden states directly 
        outputs = model.model(**inputs)
        last_hidden = outputs.last_hidden_state  # (batch=1, seq_len, hidden_dim)
        
        # Mean pool across the sequence length dimension
        embedding = last_hidden.mean(dim=1).squeeze(0)  # (hidden_dim,)
        english_embeddings_anchors.append(embedding.cpu())
        
    # Process Test Words
    for word in english_test:
        inputs = tokenizer(word, return_tensors="pt").to(device)
        
        # Optimization: same base-model forward pass
        outputs = model.model(**inputs)
        last_hidden = outputs.last_hidden_state  # (batch=1, seq_len, hidden_dim)
        
        embedding = last_hidden.mean(dim=1).squeeze(0)  # (hidden_dim,)
        english_embeddings_test.append(embedding.cpu())



Using device: mps


In [15]:
# ==========================================
# Save the English embeddings to disk
# ==========================================
# Stack list of 1D tensors into a 2D tensor
anchor_tensor = torch.stack(english_embeddings_anchors)
test_tensor = torch.stack(english_embeddings_test)

# Organize them into a dictionary
embeddings_dict = {
    "anchors": anchor_tensor,
    "anchor_words": english_anchors,
    "test": test_tensor,
    "test_words": english_test
}

# Save to disk
torch.save(embeddings_dict, "english_embeddings.pt")
print(f"\nSaved {len(english_embeddings_anchors)} anchor embeddings to 'english_embeddings.pt'!")



Saved 100 anchor embeddings to 'english_embeddings.pt'!


In [18]:
import torch.nn.functional as F
similarity_spaces_ch = {} # Each test vector gets all similarity space over 100 anchors 
for i in range(len(chinese_embeddings_test)): 
    space = []
    for e in chinese_embeddings_anchors:
       space.append(F.cosine_similarity(chinese_embeddings_test[i], e, dim=0))
    similarity_spaces_ch[i] = space

print(similarity_spaces_ch)

{0: [tensor(0.9052), tensor(0.8781), tensor(0.8404), tensor(0.8828), tensor(0.8186), tensor(0.7964), tensor(0.8959), tensor(0.8366), tensor(0.9034), tensor(0.7988), tensor(0.8787), tensor(0.8666), tensor(0.8870), tensor(0.9002), tensor(0.8730), tensor(0.9215), tensor(0.8870), tensor(0.8214), tensor(0.8042), tensor(0.8721), tensor(0.9048), tensor(0.8854), tensor(0.7334), tensor(0.8948), tensor(0.8899), tensor(0.9136), tensor(0.8214), tensor(0.8860), tensor(0.9321), tensor(0.8292), tensor(0.8333), tensor(0.8863), tensor(0.8303), tensor(0.8301), tensor(0.8444), tensor(0.8444), tensor(0.9100), tensor(0.9007), tensor(0.8968), tensor(0.8677), tensor(0.9030), tensor(0.7452), tensor(0.8669), tensor(0.8916), tensor(0.8348), tensor(0.8549), tensor(0.8860), tensor(0.8367), tensor(0.9212), tensor(0.9099), tensor(0.8625), tensor(0.8845), tensor(0.8200), tensor(0.8739), tensor(0.8984), tensor(0.8530), tensor(0.8339), tensor(0.9280), tensor(0.8748), tensor(0.9115), tensor(0.8440), tensor(0.8657), ten

In [19]:

import torch.nn.functional as F
similarity_spaces_en = {} # Each test vector gets all similarity space over 100 anchors 
for i in range(len(english_embeddings_test)): 
    space = []
    for e in english_embeddings_anchors:
       space.append(F.cosine_similarity(english_embeddings_test[i], e, dim=0))
    similarity_spaces_en[i] = space

print(similarity_spaces_en)

{0: [tensor(0.7967), tensor(0.7895), tensor(0.7715), tensor(0.7780), tensor(0.7993), tensor(0.7007), tensor(0.7574), tensor(0.7716), tensor(0.7683), tensor(0.7556), tensor(0.8143), tensor(0.7848), tensor(0.7967), tensor(0.7689), tensor(0.7619), tensor(0.8154), tensor(0.8117), tensor(0.7857), tensor(0.7282), tensor(0.7643), tensor(0.7789), tensor(0.7935), tensor(0.7923), tensor(0.8250), tensor(0.8276), tensor(0.8182), tensor(0.7831), tensor(0.7773), tensor(0.7811), tensor(0.7764), tensor(0.7461), tensor(0.7923), tensor(0.7451), tensor(0.7974), tensor(0.7918), tensor(0.7693), tensor(0.8212), tensor(0.7986), tensor(0.7967), tensor(0.6974), tensor(0.7995), tensor(0.6999), tensor(0.7388), tensor(0.8048), tensor(0.7871), tensor(0.7172), tensor(0.7746), tensor(0.7821), tensor(0.8263), tensor(0.7697), tensor(0.7605), tensor(0.7036), tensor(0.7379), tensor(0.7918), tensor(0.8133), tensor(0.7432), tensor(0.7557), tensor(0.7737), tensor(0.8038), tensor(0.8231), tensor(0.7933), tensor(0.5877), ten

In [32]:
import numpy as np
from scipy.stats import entropy


def kl_divergence(p, q):
    # Add epsilon to avoid log(0)
    epsilon = 1e-15
    p = np.asarray(p) + epsilon
    q = np.asarray(q) + epsilon
    # Renormalize
    p = p / p.sum()
    q = q / q.sum()
    return entropy(p, q) 

In [ ]:
def element_rank(arr, position):
    arr = np.asarray(arr)
    # argsort gives the rank of each element
    ranks = np.argsort(np.argsort(arr))
    return ranks[position]

SyntaxError: invalid syntax (2291892548.py, line 5)

In [ ]:
#lets try on the first word 
print(english_test[0]) 
print(chinese_test[0]) 
kl_min = float('inf')  
ranks_of_correct =[]
idx =-1
for j in range(len(english_embeddings_test)):
    curr_kl_list = []
    for i in range(len(chinese_embeddings_test)): 
        kl = kl_divergence(similarity_spaces_en[j],similarity_spaces_ch[i])
        curr_kl_list.append(kl)
    ranks_of_correct.append(element_rank(curr_kl_list, j))

market
市场


In [42]:
np.average(ranks_of_correct)

np.float64(3.7)

## Analysis & Quality Check of the Implementation

### What Went Well:
1. **Bypassing the Unembedding Layer (`.model`)**: Your forward pass directly queries the base transformer, skipping the expensive classification head (`lm_head`). This drastically reduces VRAM usage and speeds up inference.
2. **Mean-Pooling**: Taking the average across the sequence length (`.mean(dim=1)`) is the correct way to handle tokenizers that split words into a variable number of tokens.
3. **Proper CPU Offloading**: Calling `.cpu()` before appending prevents GPU memory leaks.

### Critical Data Mismatch Discovered:
During the quality check, a translation mismatch was found in your parallel test sets:
- **`english_test[7]`** was set to `"growth"` (which duplicated the previous word).
- **`chinese_test[7]`** was set to `"选择"` (which actually translates to `"choice"` or `"selection"`).

Because `"growth"` was mapped to `"选择"`, the correct rank of the 8th word was heavily penalized. Correcting `english_test[7]` to `"choice"` improves the overall alignment average rank from **3.70** down to **3.50** (lower is better, random chance is 4.5).

### Math Note on KL Divergence vs Vector Cosine Similarity:
Using KL divergence on normalized cosine similarities is creative! However, because cosine similarities fall between -1 and 1, adding epsilon and normalizing them as probability distributions (`p / p.sum()`) dampens their variance, turning them into flat distributions. 

In the original paper, the relative representations $R(x)$ are treated as standard vectors in $\mathbb{R}^k$. We can compare them directly using **Cosine Similarity of the relative vectors** or **Euclidean Distance**.

In [ ]:
# ==========================================
# Alternative: Direct Cosine Similarity of Relative Representations
# ==========================================
# 1. Convert similarity space dictionaries to standard PyTorch tensors
sim_spaces_ch = torch.stack([torch.tensor(similarity_spaces_ch[i]) for i in range(len(similarity_spaces_ch))])
sim_spaces_en = torch.stack([torch.tensor(similarity_spaces_en[i]) for i in range(len(similarity_spaces_en))])

# 2. Normalize the relative vectors to L2 norm of 1.0
sim_spaces_ch_norm = sim_spaces_ch / sim_spaces_ch.norm(dim=1, keepdim=True)
sim_spaces_en_norm = sim_spaces_en / sim_spaces_en.norm(dim=1, keepdim=True)

# 3. Compute pairwise Cosine Similarity Matrix
# cell (i, j) is the similarity between English word i and Chinese word j in the relative space
rel_sim_matrix = sim_spaces_en_norm @ sim_spaces_ch_norm.t()

# Evaluate ranks using Cosine Similarity
ranks_cos = []
for j in range(len(english_test)):
    # 1 - sim to rank smaller values first using argsort
    curr_cos = (1.0 - rel_sim_matrix[j]).numpy()
    rank = element_rank(curr_cos, j)
    ranks_cos.append(rank)
    print(f"Word: {english_test[j]} -> {chinese_test[j]} | Rank: {rank} (out of {len(english_test)-1})")

print(f"\nCosine Similarity Average Rank: {np.mean(ranks_cos):.2f}")